# 02 Data preprocessing

In [1]:
import pandas as pd
import numpy as np
import re
import html
from pathlib import Path

DATA_PATH = "../data/raw/1429_1.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

/tmp/ipykernel_25858/2557489459.py:9: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


(34660, 21)


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [2]:
df_processed = df.copy()

## 1. Missing target and text

In [3]:
# drop missing target
print("missing rating:", df_processed["reviews.rating"].isna().sum())
df_processed = df_processed.dropna(subset=["reviews.rating"])

# missing review text
print("missing text:", df_processed["reviews.text"].isna().sum())
df_processed = df_processed.dropna(subset=["reviews.text"])

df_processed["reviews.rating"] = df_processed["reviews.rating"].astype(int)

print(df_processed.shape)

missing rating: 33
missing text: 1
(34626, 21)


## 2. Remove unusable columns

In [4]:
columns_to_drop = [
    "reviews.id",
    "reviews.userCity",
    "reviews.userProvince",
    "reviews.didPurchase",
    "reviews.dateAdded",
]

df_processed = df_processed.drop(columns=columns_to_drop)
print(df_processed.shape)
print(df_processed.columns.tolist())

(34626, 16)
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateSeen', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username']


In [5]:
dup_key = ["asins", "reviews.username", "reviews.date", "reviews.text"]
print("duplicates on key columns:", df_processed.duplicated(subset=dup_key).sum())
df_processed = df_processed.drop_duplicates(subset=dup_key)
print(df_processed.shape)

duplicates on key columns: 0
(34626, 16)


In [6]:
columns_to_drop = [
    "id",
    "keys",
    "reviews.sourceURLs",
    "reviews.username",
]

df_processed = df_processed.drop(columns=columns_to_drop)
print(df_processed.shape)
print(df_processed.columns.tolist())

(34626, 12)
['name', 'asins', 'brand', 'categories', 'manufacturer', 'reviews.date', 'reviews.dateSeen', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.text', 'reviews.title']


In [7]:
#recover missing product names through `asins` 

name_by_asin = (
    df_processed.dropna(subset=["name"])
    .groupby("asins")["name"]
    .agg(lambda s: s.mode().iloc[0])
)
df_processed["product_name"] = df_processed["name"].fillna(df_processed["asins"].map(name_by_asin))
print("rows without name before recovery:", df_processed["name"].isna().sum())
print("names still missing after recovery:", df_processed["product_name"].isna().sum())

# NEW: the old name column is no longer needed
df_processed = df_processed.drop(columns=["name"])

rows without name before recovery: 6759
names still missing after recovery: 6088


In [8]:
text_metadata_columns = [
    "product_name",
    "asins",
    "brand",
    "categories",
    "manufacturer",
    "reviews.title",
]

for column in text_metadata_columns:
    df_processed[column] = df_processed[column].fillna("Unknown")

missing = df_processed.isna().sum()
print(missing[missing > 0])

reviews.date            29
reviews.doRecommend    560
reviews.numHelpful     495
dtype: int64


In [9]:
## 7. Helpful votes, recommendation and date

# numHelpful: 0 means "none or unknown"
df_processed["reviews.numHelpful"] = df_processed["reviews.numHelpful"].fillna(0)

#keep unknown recommendations visible instead of filling with False
df_processed["reviews.doRecommend"] = (
    df_processed["reviews.doRecommend"]
    .map({True: "yes", False: "no"})
    .fillna("unknown")
)
print(df_processed["reviews.doRecommend"].value_counts())

#parse the date; missing dates stay NaT (no "Unknown" text)
df_processed["reviews.date"] = pd.to_datetime(df_processed["reviews.date"], errors="coerce", utc=True)

missing = df_processed.isna().sum()
print(missing[missing > 0])   # only reviews.date should appear here
df_processed.dtypes

reviews.doRecommend
yes        32682
no          1384
unknown      560
Name: count, dtype: int64
reviews.date    89
dtype: int64


asins                                  str
brand                                  str
categories                             str
manufacturer                           str
reviews.date           datetime64[us, UTC]
reviews.dateSeen                       str
reviews.doRecommend                    str
reviews.numHelpful                 float64
reviews.rating                       int64
reviews.text                           str
reviews.title                          str
product_name                           str
dtype: object

In [10]:
## Sentiment label

In [11]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df_processed["sentiment"] = df_processed["reviews.rating"].apply(rating_to_sentiment)
df_processed[["reviews.rating", "sentiment"]].head(10)

,reviews.rating,sentiment
0,5,positive
1,5,positive
2,5,positive
3,4,positive
4,5,positive
5,5,positive
6,4,positive
7,5,positive
8,5,positive
9,5,positive


In [12]:
# validate the mapping
pd.crosstab(df_processed["reviews.rating"], df_processed["sentiment"])

sentiment,negative,neutral,positive
reviews.rating,,,
1,410,0,0
2,402,0,0
3,0,1499,0
4,0,0,8541
5,0,0,23774


In [13]:
# class distribution
print(df_processed["sentiment"].value_counts())
print(df_processed["sentiment"].value_counts(normalize=True) * 100)
# The dataset is highly imbalanced: do not judge models by accuracy alone.
# Use precision, recall, F1, macro F1 and the confusion matrix.

sentiment
positive    32315
neutral      1499
negative      812
Name: count, dtype: int64
sentiment
positive    93.325825
neutral      4.329117
negative     2.345059
Name: proportion, dtype: float64


In [14]:
print(df_processed.shape)
print(df_processed.columns.tolist())
df_processed.isna().sum()

(34626, 13)
['asins', 'brand', 'categories', 'manufacturer', 'reviews.date', 'reviews.dateSeen', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.text', 'reviews.title', 'product_name', 'sentiment']


asins                   0
brand                   0
categories              0
manufacturer            0
reviews.date           89
reviews.dateSeen        0
reviews.doRecommend     0
reviews.numHelpful      0
reviews.rating          0
reviews.text            0
reviews.title           0
product_name            0
sentiment               0
dtype: int64

##NLP preprocessing of `reviews.text`

In [15]:
def clean_text(text):
    text = str(text)

    # lowercase
    text = text.lower()

    # convert HTML entities such as &nbsp; into normal characters
    text = html.unescape(text)

    # remove HTML / unwanted markup
    text = re.sub(r"<[^>]+>", " ", text)

    # remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df_processed["clean_text"] = df_processed["reviews.text"].apply(clean_text)
df_processed[["reviews.text", "clean_text"]].head(10)

,reviews.text,clean_text
0,This product so far has not disappointed. My c...,this product so far has not disappointed. my c...
1,great for beginner or experienced person. Boug...,great for beginner or experienced person. boug...
2,Inexpensive tablet for him to use and learn on...,inexpensive tablet for him to use and learn on...
3,I've had my Fire HD 8 two weeks now and I love...,i've had my fire hd 8 two weeks now and i love...
4,I bought this for my grand daughter when she c...,i bought this for my grand daughter when she c...
5,This amazon fire 8 inch tablet is the perfect ...,this amazon fire 8 inch tablet is the perfect ...
6,"Great for e-reading on the go, nice and light ...","great for e-reading on the go, nice and light ..."
7,"I gave this as a Christmas gift to my inlaws, ...","i gave this as a christmas gift to my inlaws, ..."
8,Great as a device to read books. I like that i...,great as a device to read books. i like that i...
9,I love ordering books and reading them with th...,i love ordering books and reading them with th...


In [16]:
# empty reviews after cleaning
empty_text = df_processed["clean_text"].str.strip().eq("")
print("empty after cleaning:", empty_text.sum())

df_processed = df_processed[~empty_text].copy()
print(df_processed.shape)

empty after cleaning: 1
(34625, 14)


In [17]:
## handle duplicate texts (prevents the same review landing in train and test)

print("duplicate clean_text rows before:", df_processed["clean_text"].duplicated().sum())

# same text with different sentiment labels?
conflicts = (df_processed.groupby("clean_text")["sentiment"].nunique() > 1).sum()
print("texts with conflicting labels:", conflicts)

before = len(df_processed)
df_processed = df_processed.drop_duplicates(subset=["clean_text", "sentiment"])
print("removed:", before - len(df_processed))
print(df_processed.shape)

print(df_processed["sentiment"].value_counts())
print(df_processed["sentiment"].value_counts(normalize=True) * 100)

duplicate clean_text rows before: 11
texts with conflicting labels: 0
removed: 11
(34614, 14)
sentiment
positive    32305
neutral      1499
negative      810
Name: count, dtype: int64
sentiment
positive    93.329289
neutral      4.330618
negative     2.340094
Name: proportion, dtype: float64


In [18]:
# texts that still appear more than once (these have conflicting labels)
conflict_rows = df_processed[df_processed["clean_text"].duplicated(keep=False)].sort_values("clean_text")
conflict_rows[["reviews.text", "clean_text", "reviews.rating", "sentiment"]].head(20)

,reviews.text,clean_text,reviews.rating,sentiment


## Final columns


In [19]:
df_processed[["product_name", "brand", "categories"]].head()

,product_name,brand,categories
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."


In [20]:
final_columns = [
    "product_name",
    "asins",
    "brand",
    "categories",
    "manufacturer",
    "reviews.date",
    "reviews.doRecommend",
    "reviews.numHelpful",
    "reviews.rating",
    "reviews.title",
    "reviews.text",
    "clean_text",
    "sentiment",
]

df_processed = df_processed[final_columns].copy()
print(df_processed.shape)

(34614, 13)


## 12. Final checks 

In [21]:
missing = df_processed.isna().sum()
print("columns with missing values:")
print(missing[missing > 0])    # only reviews.date is expected

print("missing in text/target:", df_processed[["clean_text", "sentiment"]].isna().sum().sum())   # must be 0
print("duplicate rows:", df_processed.duplicated().sum())

print(df_processed["sentiment"].value_counts())
print(df_processed["reviews.rating"].value_counts().sort_index())

columns with missing values:
reviews.date    89
dtype: int64
missing in text/target: 0
duplicate rows: 0
sentiment
positive    32305
neutral      1499
negative      810
Name: count, dtype: int64
reviews.rating
1      410
2      400
3     1499
4     8537
5    23768
Name: count, dtype: int64


In [22]:
# review length after preprocessing
df_processed["text_length"] = df_processed["clean_text"].str.len()
df_processed["text_length"].describe()

count    34614.000000
mean       158.051280
std        182.143262
min          3.000000
25%         70.000000
50%        106.000000
75%        183.000000
max      10670.000000
Name: text_length, dtype: float64

## 13. Save the processed dataset

In [23]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = processed_dir / "customer_reviews_processed.csv"

df_processed.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")

Saved to: ../data/processed/customer_reviews_processed.csv
